# Train Gradient Boost
## By: Vincent Buchner


In [1]:
# Imports
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../')))

from sklearn.ensemble import VotingRegressor
from sklearn.model_selection import train_test_split
from code_files.train import train, train_in_batches, grid_search, random_search, save_model
from code_files.data_preperation import prepare_for_train
import pandas as pd
import numpy as np
import importlib

In [2]:
# Load Dataset
df_amazon = pd.read_csv("../../dataset/eda_amazon_sales_report.csv")
df_amazon.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117123 entries, 0 to 117122
Data columns (total 24 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   Unnamed: 0                           117123 non-null  int64  
 1   Size                                 117123 non-null  int64  
 2   Qty                                  117123 non-null  int64  
 3   Amount                               117123 non-null  float64
 4   promotion-ids                        117123 non-null  int64  
 5   B2B                                  117123 non-null  int64  
 6   Status_Cancelled                     117123 non-null  bool   
 7   Status_Shipped                       117123 non-null  bool   
 8   Status_Shipped - Delivered to Buyer  117123 non-null  bool   
 9   Fulfilment_Amazon                    117123 non-null  bool   
 10  Fulfilment_Merchant                  117123 non-null  bool   
 11  ship-service-

In [3]:
# Split and Prepare for train
dftrain, dftest = train_test_split(df_amazon, test_size=0.1, random_state=42)
Xtrain_prepared, ytrain_prepared, Xtest_prepared, ytest_prepared = prepare_for_train(dftrain, dftest)

### No Grid Search for Voting Regressor
We will use the VotingRegressor from sklearn.ensemble to combine the predictions of multiple models. The Voting Regressor model uses many smaller models for it's training process, and the actual model itself doesn't have many parameter to hypertune. Therefore, we just use the best models from Project #1 as our models for the Voting Regressor. These models were already hypertuned in Project #1.


In [4]:
# Train

from sklearn.linear_model import Ridge, SGDRegressor
from sklearn.tree import DecisionTreeRegressor


model = VotingRegressor(
    estimators=[
        ('Decision Tree', DecisionTreeRegressor(
            **{'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2})),
        ('Ridge', Ridge(
            **{'alpha': 0.1, 'fit_intercept': True, 'solver': 'sag', 'tol': 0.01})),
        ('SGD', SGDRegressor(**{'alpha': 0.0001, 'eta0': 0.001,
                                'learning_rate': 'invscaling', 'penalty': 'elasticnet'}))
    ]
)

model, scores = train(model, Xtrain_prepared, ytrain_prepared, Xtest_prepared, ytest_prepared)

print(f"mae: {scores[0]}, rmse: {scores[1]}, r2: {scores[2]}")

mae: 213.2108739831913, rmse: 278.24065796679884, r2: 0.03296875626826834
